# PSSP benchmark — interleaved latency and peak-memory measurements (Colab)

Companion to `PSSP_real_benchmark.ipynb`. For the five models of the paper it measures CPU and GPU latency at batch size 1 on real CB513 chains, peak GPU memory and process memory (RSS).

**Design (to control the noise of a shared cloud VM):**
* every model lives in its **own process** (memory numbers are not mixed between models);
* all model processes are started first and then measured **in interleaved rounds**: in each round every model processes the same 100 chains once, in a **random order** that changes every round, while the other processes wait idle;
* speed ratios between models are computed **within each round** (paired), so slow periods of the VM affect all models of that round alike.

Latency and memory do not depend on the trained weight values, so the classification heads are randomly initialized with the exact architectures of the paper; the ESM-2 backbones are the real pretrained models.

**Run:** `Runtime → Change runtime type → T4 GPU`, then `Runtime → Run all` (~20 min). `efficiency_interleaved.zip` is downloaded at the end.

In [ ]:
# ==== 1. Configuration ====
N_SEQ       = 100     # first CB513 chains in file order
ROUNDS      = 10      # interleaved rounds per device
WARMUP      = 3       # warm-up chains per process before the first round
CPU_THREADS = 2
MODELS = ['WindowMLP', 'Hybrid 1D-CNN-BiLSTM', 'ESM-2 (8M) Linear Probe', 'ESM-2 (8M) MLP Head', 'ESM-2 (35M) MLP Head']
OUT = '/content/efficiency_interleaved'

In [ ]:
# ==== 2. Environment and data ====
!pip -q install transformers psutil
import os, json, subprocess, sys, time, platform, random, urllib.request
import numpy as np, pandas as pd, torch, transformers
os.makedirs(OUT, exist_ok=True)
assert torch.cuda.is_available(), 'Select a GPU runtime'
sh = lambda c: subprocess.run(c, shell=True, capture_output=True, text=True).stdout.strip()
ENV = dict(gpu=torch.cuda.get_device_name(0), cpu=sh("lscpu | grep 'Model name' | sed 's/Model name:\\s*//'"), n_cpu=os.cpu_count(),
           ram_total_MB=sh("free -m | awk '/Mem:/{print $2}'"), cpu_threads_used=CPU_THREADS, torch=torch.__version__,
           transformers=transformers.__version__, python=platform.python_version(), cuda=torch.version.cuda,
           n_seq=N_SEQ, rounds=ROUNDS, warmup=WARMUP, design='interleaved rounds, random model order per round, one process per model',
           date=time.strftime('%Y-%m-%d %H:%M'))
print(json.dumps(ENV, indent=2)); json.dump(ENV, open(f'{OUT}/environment.json', 'w'), indent=2)
url = 'https://huggingface.co/datasets/proteinea/secondary_structure_prediction/resolve/main/CB513.csv'
if not os.path.exists('/content/CB513.csv'): urllib.request.urlretrieve(url, '/content/CB513.csv')
seqs = [str(s).strip().upper() for s in pd.read_csv('/content/CB513.csv')['input']][:N_SEQ]
json.dump(seqs, open('/content/bench_seqs.json', 'w'))
print(len(seqs), 'chains, mean length', round(sum(map(len, seqs)) / len(seqs), 1))

In [ ]:
%%writefile /content/bench_worker.py
# One model, one device, one process. Protocol on stdin/stdout:
#   prints "READY {...}" after loading; then for each line "RUN" times all chains once and prints "DONE {...}";
#   on "STOP" prints "FINAL {...}" with memory statistics and exits.
import sys, json, time, resource
import numpy as np, psutil, torch, torch.nn as nn, torch.nn.functional as F
name, device, threads, warmup = sys.argv[1], sys.argv[2], int(sys.argv[3]), int(sys.argv[4])
torch.set_num_threads(threads); torch.manual_seed(0)
dev = torch.device(device); proc = psutil.Process()
AA = 'ACDEFGHIKLMNPQRSTVWY'; AA_IX = {a: i + 1 for i, a in enumerate(AA)}
seqs = json.load(open('/content/bench_seqs.json'))
def emit(tag, obj): sys.stdout.write(tag + ' ' + json.dumps(obj) + '\n'); sys.stdout.flush()

class WindowMLP(nn.Module):
    def __init__(self, W=17, d=21, h=128, p=0.2):
        super().__init__(); self.W, self.half = W, W // 2
        self.net = nn.Sequential(nn.Linear(W * d, h, bias=False), nn.BatchNorm1d(h), nn.ReLU(), nn.Dropout(p),
                                 nn.Linear(h, h // 2, bias=False), nn.BatchNorm1d(h // 2), nn.ReLU(), nn.Dropout(p), nn.Linear(h // 2, 3))
    def forward(self, x, lm):
        B, L, d = x.shape
        xp = F.pad(x, (0, 0, self.half, self.half)); xp[:, :self.half, 0] = 1; xp[:, -self.half:, 0] = 1
        win = xp.unfold(1, self.W, 1).permute(0, 1, 3, 2).reshape(B, L, self.W * d)
        out = x.new_zeros(B, L, 3); out[lm] = self.net(win[lm]); return out

class HybridCNNBiLSTM(nn.Module):
    def __init__(self, d=21, conv=128, hid=128, p=0.3):
        super().__init__(); a = conv // 3; c = conv - 2 * a
        self.c3, self.c5, self.c7 = nn.Conv1d(d, a, 3, padding=1), nn.Conv1d(d, a, 5, padding=2), nn.Conv1d(d, c, 7, padding=3)
        self.bn = nn.BatchNorm1d(conv); self.sdrop = nn.Dropout1d(p)
        self.lstm = nn.LSTM(conv, hid, num_layers=2, batch_first=True, bidirectional=True, dropout=p)
        self.ln = nn.LayerNorm(2 * hid)
        self.head = nn.Sequential(nn.Linear(2 * hid, 64), nn.ReLU(), nn.Dropout(p), nn.Linear(64, 3))
    def forward(self, x, lm):
        h = x.transpose(1, 2); h = torch.cat([self.c3(h), self.c5(h), self.c7(h)], 1)
        h = self.sdrop(F.relu(self.bn(h))).transpose(1, 2)
        pk = nn.utils.rnn.pack_padded_sequence(h, lm.sum(1).cpu(), batch_first=True, enforce_sorted=False)
        o, _ = self.lstm(pk); o, _ = nn.utils.rnn.pad_packed_sequence(o, batch_first=True, total_length=x.shape[1])
        return self.head(self.ln(o))

def one_hot(seq):
    tok = torch.tensor([[AA_IX.get(a, 0) for a in seq]])
    return F.one_hot(tok, 21).float().to(dev), torch.ones(1, len(seq), dtype=torch.bool, device=dev)
def sync():
    if dev.type == 'cuda': torch.cuda.synchronize()

rss_base = proc.memory_info().rss / 2**20
if name in ('WindowMLP', 'Hybrid 1D-CNN-BiLSTM'):
    model = (WindowMLP() if name == 'WindowMLP' else HybridCNNBiLSTM()).to(dev).eval(); modules = [model]
    def run(seq):
        x, lm = one_hot(seq); return model(x, lm).argmax(-1)
else:
    from transformers import AutoTokenizer, EsmModel
    bname = 'facebook/esm2_t12_35M_UR50D' if '35M' in name else 'facebook/esm2_t6_8M_UR50D'
    tok = AutoTokenizer.from_pretrained(bname); bb = EsmModel.from_pretrained(bname, add_pooling_layer=False).to(dev).eval()
    d = bb.config.hidden_size
    head = (nn.Linear(d, 3) if 'Linear' in name else nn.Sequential(nn.Linear(d, 64), nn.ReLU(), nn.Dropout(0.2), nn.Linear(64, 3))).to(dev).eval()
    modules = [bb, head]
    def run(seq):
        s = ''.join(c if c in AA else 'X' for c in seq)[:1022]; enc = tok([s], return_tensors='pt')
        h = bb(input_ids=enc['input_ids'].to(dev), attention_mask=enc['attention_mask'].to(dev)).last_hidden_state[0, 1:-1]
        return head(h).argmax(-1)
sync()
rss_load = proc.memory_info().rss / 2**20
gpu_load = torch.cuda.memory_allocated() / 2**20 if dev.type == 'cuda' else None
with torch.no_grad():
    for s in seqs[:warmup]: run(s)
sync()
if dev.type == 'cuda': torch.cuda.reset_peak_memory_stats()
rss_infer_max = proc.memory_info().rss / 2**20
emit('READY', dict(model=name))
per_seq_all = []
for line in sys.stdin:
    cmd = line.strip()
    if cmd == 'RUN':
        times = []
        with torch.no_grad():
            for s in seqs:
                t0 = time.perf_counter(); run(s); sync(); times.append(1000 * (time.perf_counter() - t0))
                rss_infer_max = max(rss_infer_max, proc.memory_info().rss / 2**20)
        per_seq_all += times
        emit('DONE', dict(ms_per_seq=float(np.mean(times))))
    elif cmd == 'STOP':
        emit('FINAL', dict(model=name, device=device, n_params=sum(p.numel() for m in modules for p in m.parameters()),
                           weights_MB=sum(p.numel() * p.element_size() for m in modules for p in m.parameters()) / 2**20,
                           rss_baseline_MB=rss_base, rss_after_load_MB=rss_load, rss_inference_max_MB=rss_infer_max,
                           ru_maxrss_MB=resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1024,
                           gpu_after_load_MB=gpu_load,
                           gpu_peak_inference_MB=(torch.cuda.max_memory_allocated() / 2**20 if dev.type == 'cuda' else None),
                           per_seq_median=float(np.median(per_seq_all)), per_seq_q1=float(np.percentile(per_seq_all, 25)),
                           per_seq_q3=float(np.percentile(per_seq_all, 75))))
        break

In [ ]:
# ==== 3. Start all workers of one device, then measure in interleaved rounds ====
def read_tag(p, tag):
    while True:
        line = p.stdout.readline()
        if not line:
            raise RuntimeError(p.stderr.read()[-3000:])
        if line.startswith(tag + ' '): return json.loads(line[len(tag) + 1:])

rounds, finals = [], []
rng = random.Random(0)
for device in ['cpu', 'cuda']:
    workers = {}
    for m in MODELS:
        p = subprocess.Popen([sys.executable, '/content/bench_worker.py', m, device, str(CPU_THREADS), str(WARMUP)],
                             stdin=subprocess.PIPE, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True, bufsize=1)
        read_tag(p, 'READY'); workers[m] = p
    print(f'{device}: all workers ready')
    for r in range(ROUNDS):
        order = MODELS[:]; rng.shuffle(order)
        for m in order:
            workers[m].stdin.write('RUN\n'); workers[m].stdin.flush()
            res = read_tag(workers[m], 'DONE')
            rounds.append(dict(device=device, round=r, position=order.index(m), model=m, ms_per_seq=res['ms_per_seq']))
        last = {x['model']: x['ms_per_seq'] for x in rounds[-len(MODELS):]}
        print(f'{device} round {r + 1}/{ROUNDS}: ' + ' | '.join(f'{m}: {last[m]:.1f} ms' for m in MODELS))
    for m, p in workers.items():
        p.stdin.write('STOP\n'); p.stdin.flush(); finals.append(read_tag(p, 'FINAL')); p.wait()
R = pd.DataFrame(rounds); R.to_csv(f'{OUT}/rounds.csv', index=False)
F = pd.DataFrame(finals); F.to_csv(f'{OUT}/memory_and_per_sequence.csv', index=False)

In [ ]:
# ==== 4. Summaries and paired (within-round) speed ratios ====
S = R.groupby(['device', 'model'])['ms_per_seq'].agg(['mean', 'std', 'median', 'min', 'max']).reset_index()
S = S.merge(F, on=['device', 'model']); S['seq_per_s'] = 1000 / S['mean']; S['cv'] = S['std'] / S['mean']
S.to_csv(f'{OUT}/summary.csv', index=False)
pairs = [('ESM-2 (8M) MLP Head', 'Hybrid 1D-CNN-BiLSTM'), ('ESM-2 (35M) MLP Head', 'ESM-2 (8M) MLP Head'),
         ('ESM-2 (8M) MLP Head', 'WindowMLP'), ('Hybrid 1D-CNN-BiLSTM', 'WindowMLP'), ('ESM-2 (35M) MLP Head', 'WindowMLP')]
W = R.pivot_table(index=['device', 'round'], columns='model', values='ms_per_seq')
rat = []
for dev_ in ['cpu', 'cuda']:
    w = W.loc[dev_]
    for a, b in pairs:
        x = (w[a] / w[b]).values
        rat.append(dict(device=dev_, slower=a, faster=b, ratio_median=np.median(x), ratio_min=x.min(), ratio_max=x.max(),
                        ratio_q1=np.percentile(x, 25), ratio_q3=np.percentile(x, 75)))
RAT = pd.DataFrame(rat); RAT.to_csv(f'{OUT}/paired_ratios.csv', index=False)
pd.set_option('display.width', 250)
print(S[['device', 'model', 'mean', 'std', 'cv', 'median', 'per_seq_median', 'per_seq_q1', 'per_seq_q3', 'rss_inference_max_MB', 'gpu_peak_inference_MB']].round(2).to_string())
print(RAT.round(2).to_string())

In [ ]:
# ==== 5. Save and download ====
import shutil
shutil.make_archive('/content/efficiency_interleaved', 'zip', OUT)
from google.colab import files; files.download('/content/efficiency_interleaved.zip')